# E01 — Viscoelasticity & Boltzmann Superposition
*Exam tool — simplified 3-part structure. For full analysis see `01_viscoelasticity_creep.ipynb`.*

---

## Part 1 — Theory Recap

### Creep Compliance (Power-Law)
$$J(t) = J_0 + A \cdot t^n \quad [\text{1/Pa}]$$
$t$ is in **seconds**. $J_0 = 1/E_0$ is the instantaneous compliance.

### Boltzmann Superposition Principle (BSP)
$$\varepsilon(t) = \sum_{i} \Delta\sigma_i \cdot J(t - t_i)$$
**Causality rule:** any term where $t < t_i$ contributes zero — the material cannot respond before load arrives.

### WLF Time–Temperature Superposition
$$\log(a_T) = \frac{-C_1 (T - T_{\text{ref}})}{C_2 + (T - T_{\text{ref}})}$$

### Reduced Time (Shift-Table Convention)
$$t_{\text{eff}} = \Delta t \cdot \frac{\alpha_{\text{ref}}}{\alpha_T}$$
At $T > T_{\text{ref}}$: $\alpha_T < \alpha_{\text{ref}}$ $\Rightarrow$ ratio $> 1$ $\Rightarrow$ **time accelerates** (more creep in less real time).

For a multi-temperature history: $t_{\text{eff}} = \sum_i \Delta t_i \cdot \dfrac{\alpha_{\text{ref}}}{\alpha_{T_i}}$

### Parameter Table
| Symbol | Description | Unit |
|--------|-------------|------|
| $J_0$ | Instantaneous compliance $= 1/E_0$ | 1/Pa |
| $A$ | Creep coefficient | 1/(Pa·s$^n$) |
| $n$ | Creep exponent | — |
| $\Delta\sigma_i$ | Stress increment at step $i$ | Pa |
| $t_i$ | Time of load application for step $i$ | h |
| $t_{\text{eval}}$ | Time at which strain is evaluated | h |
| $\alpha_T$ | Shift factor at temperature $T$ (from table) | — |
| $C_1, C_2$ | WLF constants | — |
| $T_{\text{ref}}$ | WLF reference temperature | °C |

---

## ⚠ Common Exam Pitfalls
1. **Causality error** — including BSP terms before load is applied. Always check $t - t_i \geq 0$.
2. **Shift-factor direction** — at $T > T_{\text{ref}}$, the ratio $\alpha_{\text{ref}}/\alpha_T > 1$ (time accelerates, NOT slows down).
3. **Unit mismatch** — $J(t)$ uses $t$ in **seconds**; convert hours before computing $J$.
4. **Applying WLF outside valid range** — check that $C_2 + (T - T_{\text{ref}}) \neq 0$.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Part 2 — Problem Inputs  (edit values here)
# ═══════════════════════════════════════════════════════════════════════════════
import numpy as np

# ── Creep compliance (power-law: J(t) = J0 + A·t^n, t in seconds) ────────────
# PP:   J0=7.69e-10, A=2.5e-12,  n=0.25
# POM:  J0=3.57e-10, A=8.0e-13,  n=0.22
# PC:   J0=4.35e-10, A=6.0e-13,  n=0.20
# PA66: J0=3.33e-10, A=9.0e-13,  n=0.28
J0 = 7.69e-10   # instantaneous compliance [1/Pa]
A  = 2.5e-12    # creep coefficient
n  = 0.25       # creep exponent

# ── WLF constants (time-temperature superposition) ────────────────────────────
# PP  (Tref=23°C):  C1=8.86,  C2=101.6
# PC  (Tref=145°C): C1=22.87, C2=78.09
# ABS (Tref=100°C): C1=14.3,  C2=52.5
C1    = 8.86
C2    = 101.6
T_ref = 23.0   # WLF reference temperature [°C]

# ── Shift table (alternative to WLF; set to None to use WLF) ─────────────────
# PP example: {23: 45306.05, 40: 80.84544, 60: 0.674462}
SHIFT_TABLE = None

# ── BSP load steps: list of (Δσ [MPa], t_apply [h]) ──────────────────────────
# Worked example: 6 MPa at t=0, additional 3 MPa at t=100h, evaluate at t=500h
load_steps = [
    (6.0,   0.0),   # Δσ₁ applied at t=0
    (3.0, 100.0),   # Δσ₂ superimposed at t=100 h
]
t_eval_h = 500.0          # time at which to evaluate total strain [h]
STRAIN_LIMIT_PCT = 1.5    # allowable strain [%]

# ── TTS thermal history: list of (duration [h], temperature [°C]) ─────────────
# NOTE: WLF is valid only for T > T_ref - C2. For PP_23: T > -78.6°C (always safe).
thermal_history = [
    (500.0, 23.0),   # 500 h at reference temperature (no shift)
    (200.0, 40.0),   # 200 h at elevated temperature
]
sigma_tts_MPa = 6.0   # constant stress during TTS sub-problem [MPa]

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Part 3 — Functions
# ═══════════════════════════════════════════════════════════════════════════════

def creep_compliance(t_s, J0, A, n):
    """J(t) = J0 + A*t^n [1/Pa]. t_s in seconds; clamps negative t to zero."""
    return J0 + A * (max(float(t_s), 0.0) ** n)


def bsp_strain(t_eval_s, steps, J0, A, n):
    """Boltzmann Superposition. steps = list of (sigma_Pa, t_apply_s).
    Prints each term and returns total strain (dimensionless)."""
    print(f"  {'Term':<6} {'Δσ [MPa]':<12} {'Elapsed [h]':<14} {'J(t) [1/GPa]':<16} {'ε_term [%]':<12}")
    print(f"  {'-'*6} {'-'*12} {'-'*14} {'-'*16} {'-'*12}")
    total = 0.0
    for i, (ds, ta) in enumerate(steps):
        elapsed_s = t_eval_s - ta
        if elapsed_s < 0:
            print(f"  {i+1:<6} {ds/1e6:<12.2f} {'— not applied —':<14}  skipped (causality)")
            continue
        J_val = creep_compliance(elapsed_s, J0, A, n)
        term  = ds * J_val
        print(f"  {i+1:<6} {ds/1e6:<12.2f} {elapsed_s/3600:<14.1f} {J_val*1e9:<16.4f} {term*100:<12.4f}")
        total += term
    return total


def tts_multiplier(T_C, T_ref, shift_table=None, C1=None, C2=None):
    """Returns alpha_ref/alpha_T — the time-acceleration factor at temperature T_C.

    Table branch:  ratio = alpha_ref / alpha_T  (directly from PP_SHIFT_FACTORS)
    WLF branch:    ratio = 1 / a_T  where log(a_T) = -C1*(T-T_ref)/(C2+(T-T_ref))
                   At T > T_ref: a_T < 1  →  1/a_T > 1  →  time accelerates (correct).
                   Valid only when C2 + (T - T_ref) > 0, i.e. T > T_ref - C2.
    """
    if shift_table and T_C in shift_table and T_ref in shift_table:
        ratio = shift_table[T_ref] / shift_table[T_C]
        source = 'table'
    elif C1 is not None and C2 is not None:
        denom = C2 + (T_C - T_ref)
        if abs(denom) < 1e-9:
            raise ValueError(f'WLF denominator zero at T={T_C}°C')
        if denom < 0:
            raise ValueError(
                f'WLF invalid at T={T_C}°C: denominator C2+(T-Tref)={denom:.1f} < 0. '
                f'WLF requires T > T_ref - C2 = {T_ref - C2:.1f}°C. '
                'Use a shift table or choose temperatures in the valid range.'
            )
        log_aT = -C1 * (T_C - T_ref) / denom
        aT     = 10.0 ** log_aT
        ratio  = 1.0 / aT   # 1/aT: at T>T_ref, aT<1 → ratio>1 → time accelerates
        source = 'WLF'
    else:
        raise ValueError('Provide shift_table or WLF constants (C1, C2)')
    return ratio, source


print('Functions defined.')


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Part 3 — Execution
# ═══════════════════════════════════════════════════════════════════════════════

# --- Step 1: Material constants -----------------------------------------------
print('--- Step 1: Material Constants ---')
print(f"  {'J0 (= 1/E0) [1/Pa]':<28}: {J0:.4e}")
print(f"  {'A [1/(Pa·s^n)]':<28}: {A:.2e}")
print(f"  {'n [-]':<28}: {n}")
print(f"  {'WLF C1, C2':<28}: {C1}, {C2}  (Tref={T_ref}°C)")

# --- Step 2: BSP strain -------------------------------------------------------
print()
print('--- Step 2: BSP Strain ---')
t_eval_s  = t_eval_h * 3600.0
steps_si  = [(ds * 1e6, ta * 3600.0) for ds, ta in load_steps]
print(f"  Evaluating at t = {t_eval_h:.1f} h")
print()
eps_total = bsp_strain(t_eval_s, steps_si, J0, A, n)
print()
print(f"  {'Total BSP strain ε(t_eval)':<28}: {eps_total*100:.4f} %")

# --- Step 3: TTS reduced time -------------------------------------------------
print()
print('--- Step 3: Time-Temperature Superposition ---')
t_eff_total_s = 0.0
for dt_h, T_C in thermal_history:
    dt_s = dt_h * 3600.0
    mult, src = tts_multiplier(T_C, T_ref, shift_table=SHIFT_TABLE, C1=C1, C2=C2)
    t_eff_seg_s = dt_s * mult
    t_eff_total_s += t_eff_seg_s
    print(f"  {dt_h:.0f} h at {T_C}°C  |  α_ref/α_T = {mult:.1f}x ({src})"
          f"  →  {t_eff_seg_s/3600:.0f} h equiv. at {T_ref}°C")
print(f"  {'Total effective time [h]':<28}: {t_eff_total_s/3600:.0f} h")

# --- Step 4: Strain at effective time -----------------------------------------
print()
print('--- Step 4: Strain at Effective Time ---')
sigma_Pa = sigma_tts_MPa * 1e6
J_eff    = creep_compliance(t_eff_total_s, J0, A, n)
eps_tts  = sigma_Pa * J_eff
print(f"  {'σ [MPa]':<28}: {sigma_Pa/1e6:.2f}")
print(f"  {'t_eff [h]':<28}: {t_eff_total_s/3600:.0f}")
print(f"  {'J(t_eff) [1/GPa]':<28}: {J_eff*1e9:.4f}")
print(f"  {'ε at t_eff':<28}: {eps_tts*100:.4f} %")

In [7]:
# ═══════════════════════════════════════════════════════════════════════════════
# Part 3 — Validation
# ═══════════════════════════════════════════════════════════════════════════════

pass_bsp = eps_total * 100 <= STRAIN_LIMIT_PCT
pass_tts = eps_tts   * 100 <= STRAIN_LIMIT_PCT

# WLF denominator check for each non-reference temperature
wlf_ok = True
for _, T_C in thermal_history:
    denom = C2 + (T_C - T_ref)
    if abs(denom) < 1.0:
        wlf_ok = False

overall = pass_bsp and pass_tts and wlf_ok

print('--- VALIDATION ---')
print(f"  BSP strain    : {eps_total*100:.4f}%  <=  {STRAIN_LIMIT_PCT}%"
      f"  |  {'PASS' if pass_bsp else 'FAIL'}")
print(f"  TTS strain    : {eps_tts*100:.4f}%  <=  {STRAIN_LIMIT_PCT}%"
      f"  |  {'PASS' if pass_tts else 'FAIL'}")
print(f"  WLF denom     : {'OK (no singularity)' if wlf_ok else 'WARNING — near singularity'}"
      f"  |  {'PASS' if wlf_ok else 'WARN'}")
print(f"  OVERALL       : {'PASS' if overall else 'FAIL'}")


--- VALIDATION ---
  BSP strain    : 1.1348%  <=  1.5%  |  PASS
  TTS strain    : 294960628490872422400.0000%  <=  1.5%  |  FAIL
  WLF denom     : OK (no singularity)  |  PASS
  OVERALL       : FAIL
